In [1]:
import requests

In [2]:
data_api = "https://data-api.polymarket.com"
gamma_api = "https://gamma-api.polymarket.com"

In [ ]:
def get_market_by_slug(slug):
    """Fetch full market dict from gamma API. Returns None if not found."""
    r = requests.get(f"{gamma_api}/markets/slug/{slug}", timeout=10)
    return r.json() if r.status_code == 200 else None

def get_condition_id(slug):
    m = get_market_by_slug(slug)
    return m.get("conditionId") if m else None

In [ ]:
def get_trade_user_market(slug, wallet):
    url = f"{data_api}/trades?user={wallet}&market={get_condition_id(slug)}"
    response = requests.get(url)
    if response.status_code == 200:
        return response.json()

In [ ]:
slug = "btc-updown-15m-1773508500"
wallet = "0x63ce342161250d705dc0b16df89036c8e5f9ba9a"

In [ ]:
get_condition_id(slug)

In [7]:
get_trade_user_market(slug, wallet)

[{'proxyWallet': '0x63ce342161250d705dc0b16df89036c8e5f9ba9a',
  'side': 'BUY',
  'asset': '90914039299415839467311253861927649591838329617429425749301056033772644826176',
  'conditionId': '0xf45ee94a03accb9bff9b9af8609fb2946746acc23d51e9771d589b09ebd6c271',
  'size': 5,
  'price': 0.36,
  'timestamp': 1773509007,
  'title': 'Bitcoin Up or Down - March 14, 1:15PM-1:30PM ET',
  'slug': 'btc-updown-15m-1773508500',
  'icon': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/BTC+fullsize.png',
  'eventSlug': 'btc-updown-15m-1773508500',
  'outcome': 'Up',
  'outcomeIndex': 0,
  'name': '0x8dxd',
  'pseudonym': 'Blushing-Fine',
  'bio': '',
  'profileImage': '',
  'profileImageOptimized': '',
  'transactionHash': '0xd38a0ab9e2524f1149d8ab253c2ad3abe1db5a10fbda2a10cb448e1a17f77601'},
 {'proxyWallet': '0x63ce342161250d705dc0b16df89036c8e5f9ba9a',
  'side': 'SELL',
  'asset': '90914039299415839467311253861927649591838329617429425749301056033772644826176',
  'conditionId': '0xf45ee94a03acc

In [8]:
# Fix: WebSocketApp comes from websocket-client (not 'websocket'). Run once, then restart kernel:
%pip install websocket-client

Note: you may need to restart the kernel to use updated packages.


In [9]:
import websocket
import json
import threading
import time

In [10]:
CLOB_WS = "wss://ws-subscriptions-clob.polymarket.com/ws/market"
RTDS_WS  = "wss://ws-live-data.polymarket.com"


In [11]:
def stream_btc_price(on_price):
    """Stream live BTC price from Polymarket RTDS. Calls on_price(value) on each update."""
    def on_open(ws):
        ws.send(json.dumps({"action": "subscribe", "subscriptions": [{"topic": "crypto_prices_chainlink", "type": "*", "filters": '{"symbol":"btc/usd"}'}]}))
        def ping():
            while True:
                time.sleep(5)
                try: ws.send("PING")
                except Exception: break
        threading.Thread(target=ping, daemon=True).start()

    def on_message(ws, message):
        try:
            p = json.loads(message).get("payload", {})
            if p.get("symbol") == "btc/usd":
                on_price(p["value"])
        except (json.JSONDecodeError, KeyError):
            pass

    websocket.WebSocketApp(RTDS_WS,
        on_open=on_open, on_message=on_message,
        on_error=lambda ws, e: None, on_close=lambda ws, s, m: None,
    ).run_forever()


In [12]:
import re

VARIANT_BY_INTERVAL = {5: "fiveminute", 15: "fifteen"}


def _ws_ping(ws, interval=10):
    """Send text PING on ws every interval seconds (daemon thread)."""
    def _loop():
        while True:
            time.sleep(interval)
            try: ws.send("PING")
            except Exception: break
    threading.Thread(target=_loop, daemon=True).start()


def get_price_to_beat(slug):
    """Return PTB for slug from previous window's closePrice."""
    m = re.search(r"btc-updown-(\d+)m-", slug)
    if not m: return None
    variant = VARIANT_BY_INTERVAL.get(int(m.group(1)))
    if not variant: return None
    market = get_market_by_slug(slug)
    if not market: return None
    start_iso = (market.get("events") or [{}])[0].get("startTime", "")
    if not start_iso: return None
    r = requests.get(
        "https://polymarket.com/api/past-results",
        params={"symbol": "BTC", "variant": variant, "assetType": "crypto",
                "currentEventStartTime": start_iso, "count": 1},
        timeout=10,
    )
    if r.status_code != 200: return None
    results = r.json().get("data", {}).get("results", [])
    return results[-1].get("closePrice") if results else None


def get_active_market(interval_min=5):
    """Return info dict for the current open BTC Up/Down market."""
    from datetime import datetime
    now, step = int(time.time()), interval_min * 60
    base = (now // step) * step
    for ts in [base, base + step, base + 2 * step]:
        slug = f"btc-updown-{interval_min}m-{ts}"
        m = get_market_by_slug(slug)
        if m and not m.get("closed", True):
            start_iso = (m.get("events") or [{}])[0].get("startTime", "")
            token_ids = json.loads(m["clobTokenIds"])
            outcomes  = json.loads(m["outcomes"])
            return {
                "slug": slug,
                "question": m["question"],
                "price_to_beat": get_price_to_beat(slug),
                "token_map": dict(zip(outcomes, token_ids)),
            }
    return None


state = {"ptb": None, "btc": None, "up_bid": None, "up_ask": None, "dn_bid": None, "dn_ask": None}

def _print_state():
    ptb = f"${state['ptb']:,.2f}" if state["ptb"] is not None else "..."
    btc = f"${state['btc']:,.2f}" if state["btc"] is not None else "..."
    up  = f"bid={state['up_bid']:.3f} ask={state['up_ask']:.3f}" if state["up_bid"] is not None else "..."
    dn  = f"bid={state['dn_bid']:.3f} ask={state['dn_ask']:.3f}" if state["dn_bid"] is not None else "..."
    print(f"  PTB: {ptb:>12}  BTC: {btc:>12}  |  Up {up}  Down {dn}", flush=True)

In [13]:
def stream_market_price(on_update, interval_min=5):
    """Stream best bid/ask for Up and Down outcomes of the current active market.

    Calls on_update(outcome, bid, ask) on each best_bid_ask tick.
    outcome is 'Up' or 'Down'.
    """
    market = get_active_market(interval_min)
    if not market:
        raise RuntimeError(f"No active {interval_min}m market found")

    token_map = market["token_map"]  # {"Up": token_id, "Down": token_id}
    id_to_outcome = {v: k for k, v in token_map.items()}

    def on_open(ws):
        ws.send(json.dumps({
            "assets_ids": list(token_map.values()),
            "type": "market",
            "custom_feature_enabled": True,
        }))
        _ws_ping(ws, interval=10)

    def on_message(ws, message):
        if message == "PONG": return
        try:
            events = json.loads(message)
            if not isinstance(events, list): events = [events]
            for ev in events:
                if ev.get("event_type") != "best_bid_ask": continue
                outcome = id_to_outcome.get(ev["asset_id"])
                if not outcome: continue
                on_update(outcome, float(ev.get("best_bid", 0)), float(ev.get("best_ask", 0)))
        except (json.JSONDecodeError, ValueError, KeyError):
            pass

    websocket.WebSocketApp(CLOB_WS,
        on_open=on_open, on_message=on_message,
        on_error=lambda ws, e: None, on_close=lambda ws, s, m: None,
    ).run_forever()


In [ ]:
market = get_active_market(interval_min=15)  # change to 15 for 15-min markets
print(market['price_to_beat'])

# stream_btc_price(print)
stream_market_price(lambda outcome, bid, ask: print(f"{outcome}: bid={bid:.3f} ask={ask:.3f}"), interval_min=5)

70706.6251
Up: bid=0.230 ask=0.240
Down: bid=0.760 ask=0.770
Up: bid=0.220 ask=0.240
Down: bid=0.760 ask=0.780
Up: bid=0.220 ask=0.240
Down: bid=0.760 ask=0.780
Up: bid=0.210 ask=0.240
Down: bid=0.760 ask=0.790
Up: bid=0.210 ask=0.240
Down: bid=0.760 ask=0.790
Up: bid=0.200 ask=0.240
Down: bid=0.760 ask=0.800
Up: bid=0.190 ask=0.200
Down: bid=0.800 ask=0.810
Up: bid=0.180 ask=0.200
Down: bid=0.800 ask=0.820
Up: bid=0.170 ask=0.200
Down: bid=0.800 ask=0.830
Up: bid=0.170 ask=0.190
Down: bid=0.810 ask=0.830
Up: bid=0.170 ask=0.190
Down: bid=0.810 ask=0.830
Up: bid=0.170 ask=0.190
Down: bid=0.810 ask=0.830
Up: bid=0.170 ask=0.190
Down: bid=0.810 ask=0.830
Up: bid=0.170 ask=0.190
Down: bid=0.810 ask=0.830
Up: bid=0.170 ask=0.190
Down: bid=0.810 ask=0.830
Up: bid=0.160 ask=0.190
Down: bid=0.810 ask=0.840
Up: bid=0.160 ask=0.190
Down: bid=0.810 ask=0.840
Up: bid=0.180 ask=0.190
Down: bid=0.810 ask=0.820
Up: bid=0.160 ask=0.190
Down: bid=0.810 ask=0.840
Up: bid=0.160 ask=0.190
Down: bid=0.810

In [6]:
import re
from datetime import datetime, timedelta, timezone


SPORT_CRYPTO_CATEGORIES = frozenset({"sports", "esports", "crypto"})
# Tag slugs often mark vertical; extend if needed.
SPORT_CRYPTO_TAG_SLUGS = frozenset(
    {
        "sports",
        "esports",
        "crypto",
        "nba",
        "nfl",
        "mlb",
        "nhl",
        "ufc",
        "soccer",
        "tennis",
        "golf",
        "cricket",
        "cs2",
        "dota-2",
        "lol",
        "ncaa",
    }
)


def _norm(s):
    return (s or "").strip().lower()


def _obj_tags_sport_crypto(obj):
    if not isinstance(obj, dict):
        return False
    c, sc = _norm(obj.get("category")), _norm(obj.get("subcategory"))
    if c in SPORT_CRYPTO_CATEGORIES or sc in SPORT_CRYPTO_CATEGORIES:
        return True
    for tag in obj.get("tags") or []:
        if not isinstance(tag, dict):
            continue
        ts = _norm(tag.get("slug"))
        tl = _norm(tag.get("label"))
        if ts in SPORT_CRYPTO_TAG_SLUGS or tl in SPORT_CRYPTO_TAG_SLUGS:
            return True
        if ts.startswith(
            ("nba-", "nfl-", "mlb-", "nhl-", "cs2-", "epl-", "uefa-", "crint-", "mls-", "sea-")
        ):
            return True
    return False


def is_sport_or_crypto_market(m):
    """True if this looks like sports/esports or crypto (incl. up/down) — drop from discovery lists."""
    if m.get("sportsMarketType") or m.get("gameId"):
        return True
    if _obj_tags_sport_crypto(m):
        return True
    for ev in m.get("events") or []:
        if _obj_tags_sport_crypto(ev):
            return True
    slug, q = _norm(m.get("slug")), _norm(m.get("question"))
    if re.search(
        r"(btc|bitcoin|eth|ethereum|sol|solana|xrp|doge)[_-]updown", slug
    ) or re.search(r"updown-\d+m-\d", slug):
        return True
    if re.search(r"(bitcoin|ethereum|solana|xrp|dogecoin)\s+up\s+or\s+down", q):
        return True
    return False


def without_sport_crypto(markets):
    return [m for m in markets if not is_sport_or_crypto_market(m)]


def fetch_markets_in_end_window(start_dt, end_dt, *, closed=False, active=True, limit=100, offset=0):
    """Fetch markets whose endDate is between start_dt and end_dt."""
    params = {
        "end_date_min": start_dt.isoformat(),
        "end_date_max": end_dt.isoformat(),
        "closed": str(closed).lower(),
        "active": str(active).lower(),
        "limit": limit,
        "offset": offset,
    }
    r = requests.get(f"{gamma_api}/markets", params=params, timeout=15)
    r.raise_for_status()
    return r.json()


POLY_WEB = "https://polymarket.com"


def polymarket_event_url(market):
    """Browser URL for Polymarket (site uses /event/<slug>). Uses parent event slug when nested."""
    events = market.get("events") or []
    if events and isinstance(events[0], dict):
        s = events[0].get("slug")
        if s:
            return f"{POLY_WEB}/event/{s}"
    s = market.get("slug")
    return f"{POLY_WEB}/event/{s}" if s else None


def dedupe_markets_by_event_url(markets):
    """One row per browser URL — multi-outcome events share the same /event/... page."""
    seen = set()
    out = []
    for m in markets:
        url = polymarket_event_url(m)
        key = url if url else ("__nourl__", m.get("id"), m.get("slug"))
        if key in seen:
            continue
        seen.add(key)
        out.append(m)
    return out


def print_market_rows(markets, max_rows=20):
    print(f"Unique by event URL: {len(markets)} (showing up to {max_rows} rows)")
    for i, m in enumerate(markets[:max_rows], start=1):
        title = m.get("question") or m.get("slug") or "(no title)"
        end_date = m.get("endDate") or m.get("endDateIso")
        volume = m.get("volumeNum")
        liquidity = m.get("liquidityNum")
        url = polymarket_event_url(m)
        print(f"{i:02d}. {title}")
        if url:
            print(f"    {url}")
        else:
            print("    (no slug — search on polymarket.com)")
        print(f"    endDate={end_date} | volume={volume} | liquidity={liquidity}")


# Closing soon: markets ending in the next 2 hours
now = datetime.now(timezone.utc)
closing_soon = fetch_markets_in_end_window(
    now,
    now + timedelta(hours=2),
    closed=False,
    active=True,
    limit=200,
)
closing_soon = dedupe_markets_by_event_url(without_sport_crypto(closing_soon))
print("=== Closing Soon (next 2h, active + not closed; no sports/crypto) ===")
print_market_rows(closing_soon, max_rows=25)


# Recently closed: markets that ended in the last 2 hours
recently_closed = fetch_markets_in_end_window(
    now - timedelta(hours=2),
    now,
    closed=True,
    active=False,
    limit=200,
)
recently_closed = dedupe_markets_by_event_url(without_sport_crypto(recently_closed))
print("\n=== Recently Closed (last 2h; no sports/crypto) ===")
print_market_rows(recently_closed, max_rows=25)


=== Closing Soon (next 2h, active + not closed; no sports/crypto) ===
Unique by event URL: 19 (showing up to 25 rows)
01. Will Social Democrats win the most seats in the Danish Folketing in the 2026 Danish general election?
    https://polymarket.com/event/denmark-parliamentary-election-winner
    endDate=2026-03-24T00:00:00Z | volume=280392.546608 | liquidity=25552.22237
02. Will Mette Frederiksen be the next prime minister of Denmark after the 2026 parliamentary elections?
    https://polymarket.com/event/next-prime-minister-of-denmark-after-parliamentary-election
    endDate=2026-03-24T00:00:00Z | volume=206154.17942299964 | liquidity=10157.0596
03. Will Alberto Di Rubba win the 2026 Rovigo by-election?
    https://polymarket.com/event/rovigo-by-election-winner
    endDate=2026-03-23T23:55:00Z | volume=4898.775958999999 | liquidity=6200.21769
04. Will Giulio Centenaro win the 2026 Selvazzano Dentro by-election?
    https://polymarket.com/event/selvazzano-dentro-by-election-winner
  